In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import random
import celloracle as co
import glob
co.__version__

# visualization settings
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300

save_folder = "figures"
os.makedirs(save_folder, exist_ok=True)
os.makedirs("figures/pca_figures", exist_ok=True)

/Users/edgar/Documents/research/packages/BCELL/data/230821_1/peter_mac/conda_py_3/.conda/lib/python3.8/site-packages/loompy/bus_file.py:68: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def twobit_to_dna(twobit: int, size: int) -> str:
/Users/edgar/Documents/research/packages/BCELL/data/230821_1/peter_mac/conda_py_3/.conda/lib/python3.8/site-packages/loompy/bus_file.py:85: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-obje

In [2]:
# Set the seed
random.seed(123)  # You can use any number as the seed
# Set the seed
np.random.seed(123)  # Use the same or a different seed as needed

In [3]:
def read_parameters(file_path):
    # Check if the file exists
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {file_path}")

    # Initialize an empty dictionary to store parameters
    parameters = {}

    # Loop through each line and split key and value
    for line in lines:
        if "=" in line:
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip()
            parameters[key] = value

    return parameters


In [4]:
parameters = read_parameters("../dataset_parameters/jesse.txt")
# Specify the path to the folder you want to loop through
output_data_path = os.path.join(parameters['dataset_location'],"cellOracle/data")
os.makedirs(output_data_path,exist_ok=True)
os.makedirs(os.path.join(output_data_path,"GRN"),exist_ok=True)
h5ad_folder = os.path.join(parameters['dataset_location'],parameters['h5ad_cluster_based_files'])



In [5]:
# Check the 'species' parameter and load the corresponding base GRN
if parameters['species'] == "human":
    base_GRN = co.data.load_human_promoter_base_GRN()
elif parameters['species'] == "mouse":
    base_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()
else:
    raise ValueError("Error: 'species' parameter must be 'human' or 'mouse'.")


In [6]:
%%capture
for file in glob.glob(os.path.join(h5ad_folder,"*.h5ad")):
    print(file)
    adata = sc.read_h5ad(file)

    cluster_name = adata.obs["cluster"].unique()[0]
  
    # Only consider genes with more than 1 count
    sc.pp.filter_genes(adata, min_counts=1)

    # Normalize gene expression matrix with total UMI count per cell
    sc.pp.normalize_per_cell(adata, key_n_counts='n_counts_all')

    # Select top 2000 highly-variable genes
    filter_result = sc.pp.filter_genes_dispersion(adata.X,
                                                #flavor='cell_ranger',
                                                n_top_genes=3000,
                                                log=False)

    # Subset the genes
    adata = adata[:, filter_result.gene_subset]

    # Renormalize after filtering
    sc.pp.normalize_per_cell(adata)
    # keep raw cont data before log transformation
    adata.raw = adata
    adata.layers["raw_count"] = adata.raw.X.copy()

    # Log transformation and scaling
    sc.pp.log1p(adata)
    sc.pp.scale(adata)
    # PCA
    sc.tl.pca(adata, svd_solver='arpack')
    #filename = f"{cluster_name}.png"
    #sc.pl.pca(adata,color = "condition", save=filename,title=cluster_name)

    # Instantiate Oracle object
    oracle = co.Oracle()

    # In this notebook, we use the unscaled mRNA count for the nput of Oracle object.
    adata.X = adata.layers["raw_count"].copy()

    # Instantiate Oracle object.
    oracle.import_anndata_as_raw_count(adata=adata,
                                    cluster_column_name="cluster",
                                    embedding_name="X_pca")
    
    # You can load TF info dataframe with the following code.
    oracle.import_TF_data(TF_info_matrix=base_GRN)
    # Perform PCA
    oracle.perform_PCA()

    # Select important PCs
    plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
    n_comps = np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_))>0.002))[0][0]
    plt.axvline(n_comps, c="k")
    #plt.show()
    
    n_comps = min(n_comps, 50)

    n_cell = oracle.adata.shape[0]
    print(f"cell number is :{n_cell}")
    k = int(0.025*n_cell)
    print(f"Auto-selected k is :{k}")

    oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, b_sight=k*8,
                      b_maxl=k*4, n_jobs=6)
    
    links = oracle.get_links(
        cluster_name_for_GRN_unit="cluster", 
        alpha=10,
        verbose_level=10)
    
    filename = cluster_name + ".csv"
 
    # Save as csv
    links.links_dict[cluster_name].to_csv(os.path.join(output_data_path,"GRN",filename))
    links.filter_links(p=0.001, weight="coef_abs", threshold_number=2000)
    links.plot_scores_as_rank(cluster=cluster_name, n_gene=30, save=f"{save_folder}/ranked_score")


